# Correção — Exercício Aula 06
## Visualização de Dados, Storytelling e NoSQL

**Curso:** Ciência de Dados | **Instituição:** Univassouras  
**Prof. Mesc. Diego Ramos Inácio**

---

| Parte | Tema | Exercícios |
|-------|------|------------|
| 1 | Visualização de Dados e Storytelling | Ex. 1 a 5 |
| 2 | NoSQL e MongoDB (pymongo) | Ex. 6 a 9 |

> Execute cada célula com **Shift+Enter** de cima para baixo.

`%pip install numpy pandas matplotlib seaborn pymongo`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='tab10')

np.random.seed(42)
print('Ambiente pronto!')

---
# PARTE 1 — Visualização de Dados e Storytelling

## Exercício 1 — Gráfico de Barras Comparativo

Uma rede de lojas quer comparar as vendas de 2023 e 2024 por região.  
Crie um **gráfico de barras duplas** (lado a lado) com destaque na região de maior crescimento.

In [ ]:
# Dados
regioes     = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
vendas_2023 = [85,  120,  95, 310, 175]
vendas_2024 = [102, 118, 130, 340, 210]

crescimento = [(v24 - v23) / v23 * 100
               for v23, v24 in zip(vendas_2023, vendas_2024)]

print('Crescimento por região (%):')
for r, c in zip(regioes, crescimento):
    print(f'  {r:<15} {c:+.1f}%')

In [ ]:
idx_max = crescimento.index(max(crescimento))

x       = np.arange(len(regioes))
largura = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

# barras de 2023 (cinza)
ax.bar(x - largura/2, vendas_2023, largura,
       label='2023', color='#bdc3c7')

# barras de 2024 — destaque verde na região com maior crescimento
cores_2024 = ['#2ecc71' if i == idx_max else '#3498db'
              for i in range(len(regioes))]
ax.bar(x + largura/2, vendas_2024, largura,
       label='2024', color=cores_2024)

# título, eixos e legenda
ax.set_title('Vendas por Região — 2023 vs 2024 (R$ mil)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Região')
ax.set_ylabel('Vendas (R$ mil)')
ax.legend()

# ticks com nomes das regiões
ax.set_xticks(x)
ax.set_xticklabels(regioes)

# anotação no maior crescimento
ax.annotate(
    f'+{crescimento[idx_max]:.0f}%\nMaior crescimento!',
    xy=(x[idx_max] + largura/2, vendas_2024[idx_max]),
    xytext=(x[idx_max] + largura/2 + 0.4, vendas_2024[idx_max] + 12),
    fontsize=9, color='#27ae60', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#27ae60')
)

plt.tight_layout()
plt.show()

## Exercício 2 — Gráfico de Linha com Anotação

Visualize a evolução de clientes ativos e receita ao longo de 12 meses  
em dois subplots lado a lado. Anote o mês de pico no gráfico de receita.

In [ ]:
# Dados
meses    = ['Jan','Fev','Mar','Abr','Mai','Jun',
            'Jul','Ago','Set','Out','Nov','Dez']
clientes = [1200,1350,1280,1500,1620,1580,
            1700,1850,1780,1900,2100,2050]
receita  = [48, 54, 51, 60, 65, 63,
            68, 74, 71, 76, 84, 82]

val_pico = max(receita)
idx_pico = receita.index(val_pico)
print(f'Mês de pico: {meses[idx_pico]} — R$ {val_pico} mil')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot esquerdo: clientes
axes[0].plot(meses, clientes, marker='o',
             color='#0d9488', linewidth=2, markersize=7)
axes[0].fill_between(range(12), clientes,
                     alpha=0.1, color='#0d9488')
axes[0].set_title('Clientes Ativos')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('Clientes')
axes[0].set_xticks(range(12))
axes[0].set_xticklabels(meses, rotation=45)

# Subplot direito: receita + anotação do pico
axes[1].plot(meses, receita, marker='o',
             color='#e05c2a', linewidth=2, markersize=7)
axes[1].set_title('Receita Mensal (R$ mil)')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('R$ mil')
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(meses, rotation=45)
axes[1].annotate(
    f'Pico: R$ {val_pico} mil',
    xy=(idx_pico, val_pico),
    xytext=(idx_pico - 3, val_pico - 8),
    fontsize=9, color='#c0392b', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#c0392b')
)

# Título geral
fig.suptitle('Performance 2024 — Clientes e Receita',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Exercício 3 — Distribuição com Seaborn

Compare a distribuição salarial de dois departamentos com **boxplot** e **histograma com KDE**.

In [ ]:
# Dados
np.random.seed(10)
df_salarios = pd.DataFrame({
    'salario': np.concatenate([
        np.random.normal(6500, 800, 80),
        np.random.normal(5200, 300, 80),
    ]).clip(2500, 15000),
    'depto': ['TI'] * 80 + ['Vendas'] * 80
})
print(df_salarios.groupby('depto')['salario'].describe().round(0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Subplot esquerdo: boxplot comparativo
sns.boxplot(x='depto', y='salario', data=df_salarios,
            palette={'TI': '#3498db', 'Vendas': '#e05c2a'},
            ax=axes[0])
axes[0].set_title('Salários por Departamento — Boxplot')
axes[0].set_xlabel('Departamento')
axes[0].set_ylabel('Salário (R$)')

# Subplot direito: histograma com KDE, separado por departamento
sns.histplot(data=df_salarios, x='salario', hue='depto',
             kde=True, bins=20, alpha=0.5,
             palette={'TI': '#3498db', 'Vendas': '#e05c2a'},
             ax=axes[1])
axes[1].set_title('Distribuição de Salários')
axes[1].set_xlabel('Salário (R$)')

plt.tight_layout()
plt.show()

# Resposta
std_ti    = df_salarios[df_salarios['depto'] == 'TI']['salario'].std()
std_venda = df_salarios[df_salarios['depto'] == 'Vendas']['salario'].std()
print(f'Desvio padrão  TI: R$ {std_ti:.0f}')
print(f'Desvio padrão Vendas: R$ {std_venda:.0f}')
print()
print('→ TI tem maior variabilidade.')
print('  No boxplot: caixa mais alta (Q1 a Q3 mais distantes).')
print('  No histograma: curva mais larga e achatada vs Vendas mais estreita e concentrada.')

## Exercício 4 — Heatmap de Correlação

Crie um heatmap de correlação para métricas de campanhas de marketing  
e identifique os pares com maior correlação positiva e negativa.

In [ ]:
# Dados
np.random.seed(5)
n = 120
investimento = np.random.uniform(1000, 20000, n)

df_camp = pd.DataFrame({
    'investimento_R$':   investimento,
    'cliques':           (investimento * 0.8  + np.random.normal(0, 1500, n)).clip(0),
    'conversoes':        (investimento * 0.05 + np.random.normal(0, 200,  n)).clip(0),
    'custo_por_clique':  (1 / (investimento / 5000) + np.random.normal(0, 0.3, n)).clip(0.1),
    'taxa_rejeicao_%':   (80 - investimento / 500 + np.random.normal(0, 5, n)).clip(10, 95),
})
print(df_camp.describe().round(1))

In [ ]:
# 1. Calcule a matriz de correlação
corr = df_camp.corr()

# 2. Plote o heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlação entre Métricas de Campanha',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Encontrar pares automaticamente
mask    = np.triu(np.ones_like(corr, dtype=bool))
corr_tri = corr.mask(mask)
max_par  = corr_tri.stack().idxmax()
min_par  = corr_tri.stack().idxmin()

print(f'Maior correlação positiva: {max_par[0]}  ×  {max_par[1]}')
print(f'  valor: {corr.loc[max_par]:.2f}')
print()
print(f'Maior correlação negativa: {min_par[0]}  ×  {min_par[1]}')
print(f'  valor: {corr.loc[min_par]:.2f}')

## Exercício 5 — Storytelling Completo

Monte um painel de 3 gráficos para a diretoria mostrando que  
**março foi uma anomalia pontual** e o negócio se recuperou em seguida.

In [ ]:
# Dados
meses_s1   = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun']
receita_s1 = [80,  95,  70, 110, 130, 125]
ticket_s1  = [200, 211, 184, 212, 217, 219]
pagamentos = {
    'Cartão Crédito': 45,
    'Pix':            30,
    'Boleto':         15,
    'Cartão Débito':  10
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gráfico 1: barras de receita — março em vermelho
cores = ['#e74c3c' if m == 'Mar' else '#3498db' for m in meses_s1]
axes[0].bar(meses_s1, receita_s1, color=cores)
axes[0].set_title('Receita Mensal (R$ mil)')
axes[0].set_ylabel('R$ mil')
for i, v in enumerate(receita_s1):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9)
axes[0].annotate(
    'Queda em Março\n(−26% vs Fev)',
    xy=(2, 70), xytext=(3.2, 62),
    fontsize=8.5, color='#e74c3c', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#e74c3c')
)

# Gráfico 2: linha do ticket médio
axes[1].plot(meses_s1, ticket_s1, marker='o',
             color='#0d9488', linewidth=2, markersize=8)
axes[1].set_title('Ticket Médio (R$/pedido)')
axes[1].set_ylabel('R$')
axes[1].set_ylim(150, 250)
for i, v in enumerate(ticket_s1):
    axes[1].text(i, v + 4, str(v), ha='center', fontsize=8.5)

# Gráfico 3: formas de pagamento (barh)
labels  = list(pagamentos.keys())
valores = list(pagamentos.values())
cores_p = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
axes[2].barh(labels, valores, color=cores_p)
axes[2].set_title('Formas de Pagamento (%)')
axes[2].set_xlabel('% dos pedidos')
for i, v in enumerate(valores):
    axes[2].text(v + 0.5, i, f'{v}%', va='center', fontsize=9)

fig.suptitle(
    'E-commerce — Performance 1º Semestre 2024\n'
    'Março foi uma anomalia pontual; negócio retomou crescimento em abril.',
    fontsize=12, fontweight='bold'
)

plt.tight_layout()
plt.show()

---
# PARTE 2 — NoSQL e MongoDB

Conectamos a um servidor **MongoDB real** via `pymongo` (`mongodb://localhost:27017`).

In [ ]:
# Conexão real ao MongoDB — execute antes dos exercícios 6-9
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017')
db     = client['aula06_nosql']

livros_col  = db['livros']
pedidos_col = db['pedidos']

# Limpa as coleções para permitir reexecutar o notebook do zero
livros_col.delete_many({})
pedidos_col.delete_many({})

print('Conectado ao MongoDB real!')

## Exercício 6 — Insert: Catálogo de Livros

Insira livros na coleção usando `insert_one()` e `insert_many()`.  
Cada documento deve ter: `titulo`, `autor`, `ano`, `genero`, `nota`, `disponivel`.

In [ ]:
# insert_one — um livro por vez
livros_col.insert_one({
    'titulo':     'O Hobbit',
    'autor':      'J.R.R. Tolkien',
    'ano':        1937,
    'genero':     'Fantasia',
    'nota':       9.1,
    'disponivel': True
})

# insert_many — vários de uma vez
livros_col.insert_many([
    {'titulo':'1984',         'autor':'George Orwell',    'ano':1949, 'genero':'Distopia',  'nota':9.3, 'disponivel':True},
    {'titulo':'Dom Casmurro', 'autor':'Machado de Assis', 'ano':1899, 'genero':'Romance',   'nota':8.7, 'disponivel':True},
    {'titulo':'Sapiens',      'autor':'Yuval Harari',     'ano':2011, 'genero':'História',  'nota':8.9, 'disponivel':False},
    {'titulo':'O Alquimista', 'autor':'Paulo Coelho',     'ano':1988, 'genero':'Filosofia', 'nota':7.5, 'disponivel':True},
])

print(f'\nTotal de livros na coleção: {livros_col.count_documents({})}')

## Exercício 7 — Find: Buscas e Filtros

Execute 5 consultas na coleção de livros usando diferentes filtros e operadores.

In [ ]:
print('=== 1. Todos os livros (título + nota) ===')
for l in livros_col.find({}, {'titulo': 1, 'nota': 1, '_id': 0}):
    print(f"  {l['titulo']:<25} nota: {l['nota']}")

print('\n=== 2. Livros com nota > 8.0 ===')
for l in livros_col.find({'nota': {'$gt': 8.0}}):
    print(f"  {l['titulo']} — {l['nota']}")

print('\n=== 3. Gênero Fantasia ===')
for l in livros_col.find({'genero': 'Fantasia'}):
    print(f"  {l['titulo']} ({l['ano']})")

print('\n=== 4. Livro com maior nota ===')
melhor = sorted(livros_col.find(), key=lambda x: x['nota'], reverse=True)[0]
print(f"  {melhor['titulo']} — nota: {melhor['nota']}")

print('\n=== 5. Quantidade de livros disponíveis ===')
print(f"  {livros_col.count_documents({'disponivel': True})} livros disponíveis")

## Exercício 8 — Update e Delete

Atualize campos, marque documentos e remova o livro mais antigo da coleção.

In [ ]:
print('=== 1. Atualizar nota do Dom Casmurro para 9.0 ===')
livros_col.update_one(
    {'titulo': 'Dom Casmurro'},
    {'$set': {'nota': 9.0}}
)
doc = livros_col.find_one({'titulo': 'Dom Casmurro'})
print(f"  Nova nota: {doc['nota']}")

print('\n=== 2. Adicionar campo "editora" em todos ===')
livros_col.update_many(
    {},
    {'$set': {'editora': 'Editora Brasil'}}
)

print('\n=== 3. Marcar indisponíveis os livros com nota < 8.0 ===')
livros_col.update_many(
    {'nota': {'$lt': 8.0}},
    {'$set': {'disponivel': False}}
)

print('\n=== 4. Deletar o livro mais antigo ===')
todos     = list(livros_col.find())
menor_ano = min(l['ano'] for l in todos)
print(f"  Menor ano encontrado: {menor_ano}")
livros_col.delete_one({'ano': menor_ano})

print(f'\nTotal restante: {livros_col.count_documents({})} livros')

## Exercício 9 — Integração: MongoDB + Pandas + Visualização

Fluxo completo: inserir pedidos no MongoDB → consultar → DataFrame → gráfico.

In [ ]:
# Inserir pedidos
pedidos_col.insert_many([
    {'produto':'Notebook Pro',   'categoria':'Eletrônicos', 'qtd':2,  'preco':3500.0, 'mes':'Jan'},
    {'produto':'Mouse Gamer',    'categoria':'Eletrônicos', 'qtd':5,  'preco':250.0,  'mes':'Jan'},
    {'produto':'Camisa Polo',    'categoria':'Vestuário',   'qtd':8,  'preco':89.9,   'mes':'Jan'},
    {'produto':'Tênis Running',  'categoria':'Vestuário',   'qtd':3,  'preco':299.0,  'mes':'Fev'},
    {'produto':'Cafeteira',      'categoria':'Casa',        'qtd':4,  'preco':180.0,  'mes':'Fev'},
    {'produto':'Monitor 4K',     'categoria':'Eletrônicos', 'qtd':1,  'preco':2800.0, 'mes':'Fev'},
    {'produto':'Livro Python',   'categoria':'Educação',    'qtd':12, 'preco':79.9,   'mes':'Mar'},
    {'produto':'Cadeira Gamer',  'categoria':'Casa',        'qtd':2,  'preco':1200.0, 'mes':'Mar'},
    {'produto':'Fone Bluetooth', 'categoria':'Eletrônicos', 'qtd':6,  'preco':350.0,  'mes':'Mar'},
    {'produto':'Mochila Viagem', 'categoria':'Vestuário',   'qtd':4,  'preco':220.0,  'mes':'Abr'},
    {'produto':'Curso Online',   'categoria':'Educação',    'qtd':20, 'preco':149.9,  'mes':'Abr'},
    {'produto':'Teclado Mec.',   'categoria':'Eletrônicos', 'qtd':3,  'preco':480.0,  'mes':'Abr'},
])
print(f'Pedidos inseridos: {pedidos_col.count_documents({})}')

In [ ]:
# 1. Buscar todos e carregar em DataFrame
todos      = list(pedidos_col.find({}, {'_id': 0}))
df_pedidos = pd.DataFrame(todos)
print('DataFrame carregado do MongoDB:')
print(df_pedidos.head())

# 2. Calcular receita total por pedido
df_pedidos['receita'] = df_pedidos['qtd'] * df_pedidos['preco']

# 3. Agrupar por categoria (crescente para barh)
receita_por_cat = (
    df_pedidos
    .groupby('categoria')['receita']
    .sum()
    .sort_values(ascending=True)
)
print('\nReceita por categoria:')
print(receita_por_cat.map('R$ {:,.0f}'.format))

# 4. Gráfico de barras horizontais
cores = ['#3498db', '#0d9488', '#e05c2a', '#f39c12']
fig, ax = plt.subplots(figsize=(9, 5))

receita_por_cat.plot(kind='barh', ax=ax, color=cores, width=0.6)

ax.set_title('Receita Total por Categoria — E-commerce (Jan–Abr)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Receita (R$)')
ax.set_ylabel('')
ax.set_xlim(0, receita_por_cat.max() * 1.18)

for i, v in enumerate(receita_por_cat.values):
    ax.text(v + 50, i, f'R$ {v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nCategoria com maior receita: {receita_por_cat.idxmax()} "
      f"— R$ {receita_por_cat.max():,.0f}")

---
## Pontuação sugerida

| Exercício | Pontos | Tema |
|-----------|--------|------|
| Ex. 1 | 1,0 | Barras duplas com destaque e anotação |
| Ex. 2 | 1,5 | Linhas + marcadores + anotação de pico |
| Ex. 3 | 1,5 | Boxplot + Histplot + interpretação escrita |
| Ex. 4 | 1,5 | Heatmap + identificação de pares |
| Ex. 5 | 2,0 | Storytelling: 3 gráficos integrados |
| Ex. 6 | 0,5 | Insert one + many + count |
| Ex. 7 | 1,0 | Find com filtros e operadores |
| Ex. 8 | 0,5 | Update + Delete |
| Ex. 9 | 1,5 | Integração MongoDB → Pandas → gráfico |
| **Total** | **11,0** | *(normalize para 10 se necessário)* |

---
**Prof. Mesc. Diego Ramos Inácio · Ciência de Dados · Univassouras · 2026**